本案例用于课程“智能感知与自动驾驶技术”课程的示例教学，环境配置：



In [ ]:
# 基本环境配置
# Python 3.11
# !pip install opencv-python-headless
# !pip install opencv-python
# !pip install opencv-contrib-python

加载一张图像，OpenCV的cv2.imshow()函数用于显示图像。

In [ ]:
import cv2
import numpy as np

# 读取图像
# OpenCV 默认以 BGR 格式读取图像
img_path = r'D:\AIR\Teaching\solidWhiteRight.jpg'
img_bgr_original = cv2.imread(img_path) # 保留一个原始副本

# 检查图像是否成功加载
if img_bgr_original is None:
    print("错误：无法加载图像，请检查路径 'image.jpg' 是否正确。")
    exit()

# 直接使用 OpenCV 显示 BGR 图像
cv2.imshow('Original Image (BGR)', img_bgr_original)
cv2.waitKey(0) # 等待按键
cv2.destroyAllWindows() # 关闭所有 OpenCV 窗口


现在，我们有了img_bgr_original,可以开始我们的处理之旅了。为了在后续步骤中添加文本而不修改原始图像，我们通常会操作其副本。

辅助函数：用于调整大小和添加文本以便于并排显示

为了方便地将多个图像（可能尺寸不同，或包含灰度图）并排显示在一个窗口中，并给它们加上标签，我们定义一个辅助函数。

In [ ]:
def prepare_for_display(image, text, target_width=300, target_height=250):
    """
    调整图像大小，将其转换为BGR（如果需要），并添加文本。
    """
    img_display = image.copy()

    # 如果是灰度图，转换为BGR以便与其他彩色图堆叠并显示彩色文本
    if len(img_display.shape) == 2:
        img_display = cv2.cvtColor(img_display, cv2.COLOR_GRAY2BGR)

    # 调整大小
    img_display = cv2.resize(img_display, (target_width, target_height))

    # 添加文本
    cv2.putText(img_display, text, (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA) # 绿色文本
    return img_display


# 一、颜色空间转换（cv2.cvtColor()）

计算机以数字方式表示颜色，而不同的“颜色空间”提供了不同的表示方法，适用于不同的任务。

BGR/RGB：最常见的颜色空间，由蓝(Blue)、绿(Green)、红(Red)三个通道组成。（OpenCV默认使用BGR）

Grayscale（灰度图）:只有亮度信息，没有颜色信息。简化图像，常用于边缘检测、特征提取等。

HSV（Hue,Saturation,Value）:Hue（色调），Saturation(饱和度),Value(明度)。HSV空间对于颜色分割和识别特定颜色的物体非常有用。

原理：

颜色空间转化是通过特定的数学公式将像素值从一个空间的表示映射到另一个空间的表示。例如，从BGR到灰度的转换通常是加权平均：

Gray = 0.114B + 0.587G + 0.299*R (OpenCV BGR顺序的系数)。


In [ ]:
img_bgr = img_bgr_original.copy() # 使用副本

# 1. BGR 转换为灰度图
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# 2. BGR 转换为 HSV
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

# 准备用于显示的图像
disp_original = prepare_for_display(img_bgr, "Original BGR")
disp_gray = prepare_for_display(img_gray, "Grayscale")
# HSV 图像本身不易直接观察，通常会将其转回BGR或单独看其通道
# 为了在此并排显示，我们将HSV转回BGR
disp_hsv_as_bgr = prepare_for_display(cv2.cvtColor(img_hsv, cv2.COLOR_HSV2BGR), "HSV (as BGR)")

#水平堆叠图像
combined_display = np.hstack((disp_original, disp_gray, disp_hsv_as_bgr))

cv2.imshow('Color Space Conversions', combined_display)
cv2.waitKey(0)
cv2.destroyAllWindows()


## 二、几何变换

几何变换改变图像中像素的空间位置，但不改变像素值本身。
## 1.缩放(cv2.resize())

改变图像的大小。

原理：缩放时，如果新尺寸与原尺寸不成整数倍，就需要“插值”来决定新像素的值。常见的插值方法有cv2.INTER_NEAREST, cv2.INTER_LINEAR (默认), cv2.INTER_CUBIC。

In [ ]:
img_bgr = img_bgr_original.copy()
height, width = img_bgr.shape[:2]


#cv2.imshow('Image Orig', img_bgr)

# 缩小到一半 (指定目标尺寸 dsize)
img_scaled_half = cv2.resize(img_bgr, (width // 2, height // 2), interpolation=cv2.INTER_LINEAR)
#cv2.imshow('Image Scaling half', img_scaled_half)

# 放大两倍 (使用缩放因子 fx, fy)
img_scaled_double = cv2.resize(img_bgr, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
#cv2.imshow('Image Scaling double', img_scaled_double)

# 准备显示 (固定高度，宽度自适应)
def prep_scale(image, text, target_h=270):
    h, w = image.shape[:2]
    scale = target_h / h
    resized_img = cv2.resize(image.copy(), (int(w * scale), target_h))
    cv2.putText(resized_img, text, (5,15), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,0),1)
    return resized_img

disp_s_orig = prep_scale(img_bgr, f'Original ({width}x{height})')
disp_s_half = prep_scale(img_scaled_half, f'Half ({img_scaled_half.shape[1]}x{img_scaled_half.shape[0]})')
disp_s_double = prep_scale(img_scaled_double, f'Double ({img_scaled_double.shape[1]}x{img_scaled_double.shape[0]})')

combined_scaling = np.hstack((disp_s_orig, disp_s_half, disp_s_double))
cv2.imshow('Image Scaling', combined_scaling)
cv2.waitKey(0)
cv2.destroyAllWindows()


## 2.平移

将图像在X和Y方向上移动。

原理：平移需要一个2x3的变换矩阵 M = [[1, 0, tx], [0, 1, ty]]。tx 是水平位移，ty 是垂直位移。使用 cv2.warpAffine() 应用变换。

In [ ]:
img_bgr = img_bgr_original.copy()
rows, cols = img_bgr.shape[:2]
tx = 50  # 水平平移50像素
ty = 100 # 垂直平移100像素

M_translate = np.float32([[1, 0, tx], [0, 1, ty]])
img_translated = cv2.warpAffine(img_bgr, M_translate, (cols, rows))

disp_t_orig = prepare_for_display(img_bgr, "Original")
disp_t_translated = prepare_for_display(img_translated, f"Translated (tx={tx}, ty={ty})")

combined_translation = np.hstack((disp_t_orig, disp_t_translated))
cv2.imshow('Image Translation', combined_translation)
cv2.waitKey(0)
cv2.destroyAllWindows()


## 3.旋转

围绕一个中心点旋转图像。

原理：使用cv2.getRotationMatrix2D()构建旋转矩阵，需要旋转中心、角度和缩放因子。同样使用cv2.warpAffine()应用变换。

In [ ]:
img_bgr = img_bgr_original.copy()
rows, cols = img_bgr.shape[:2]
center = (cols // 2, rows // 2) # 图像中心点
angle = 90 # 旋转角度
scale = 1.0 # 缩放因子

M_rotate = cv2.getRotationMatrix2D(center, angle, scale) # 计算旋转矩阵
img_rotated = cv2.warpAffine(img_bgr, M_rotate, (cols, rows)) 

disp_r_orig = prepare_for_display(img_bgr, "Original") 
disp_r_rotated = prepare_for_display(img_rotated, f"Rotated {angle} deg")

combined_rotation = np.hstack((disp_r_orig, disp_r_rotated))
cv2.imshow('Image Rotation', combined_rotation)
cv2.waitKey(0)
cv2.destroyAllWindows()


## 4.仿射变换

保持图像中的平行线关系。由原图中三个点及其在目标图像中对应的三个点确定。（如果你没看懂仿射变换在干什么，或者能干什么，可以看我的另一篇博客有解释）

原理：
使用 cv2.getAffineTransform() 从三对点计算2x3的变换矩阵 M。

In [ ]:
img_bgr = img_bgr_original.copy()
rows, cols = img_bgr.shape[:2] # 保持原图像大小
# 原图中的三个点
pts1_affine = np.float32([[50, 50], [cols-50, 50], [50, rows-50]]) 
 # 目标图像中的三个点
pts2_affine = np.float32([[50, 200], [cols-100, 80], [80, rows-20]])

img_bgr_with_pts = img_bgr.copy()  #副本上画点
for pt in pts1_affine:
    cv2.circle(img_bgr_with_pts, tuple(pt.astype(int)), 5, (0, 255, 0), -1) 
# 计算仿射变换矩阵
M_affine = cv2.getAffineTransform(pts1_affine, pts2_affine) 
# 应用仿射变换
img_affine_transformed = cv2.warpAffine(img_bgr, M_affine, (cols, rows)) 

disp_aff_orig = prepare_for_display(img_bgr_with_pts, "Original with Points")
disp_aff_transformed = prepare_for_display(img_affine_transformed, "Affine Transformed")

combined_affine = np.hstack((disp_aff_orig, disp_aff_transformed))
cv2.imshow('Affine Transformation', combined_affine)
cv2.waitKey(0)
cv2.destroyAllWindows()


## 5.透视变换

可将图像的一个四边形区域映射到另一个四边形，不保持平行线关系。（如果没看懂，可以看我另一篇博客）

原理：使用 cv2.getPerspectiveTransform() 从四对点计算3x3的变换矩阵 M。

In [ ]:
img_bgr = img_bgr_original.copy()
rows, cols = img_bgr.shape[:2]

# 这些点需要你根据你的图像手动选取或通过其他方法检测
# 这里使用图像的角点和一些偏移作为示例
pts1_perspective = np.float32([
    [cols*0.1, rows*0.1], [cols*0.9, rows*0.2],
    [cols*0.2, rows*0.9], [cols*0.8, rows*0.8]
])

img_bgr_with_pts_persp = img_bgr.copy()
for pt in pts1_perspective:
    cv2.circle(img_bgr_with_pts_persp, tuple(pt.astype(int)), 5, (0, 0, 255), -1) # 红点

output_width = 300
output_height = 400
pts2_perspective = np.float32([
    [0, 0], [output_width - 1, 0],
    [0, output_height - 1], [output_width - 1, output_height - 1]
])

M_perspective = cv2.getPerspectiveTransform(pts1_perspective, pts2_perspective)
img_perspective_transformed = cv2.warpPerspective(img_bgr, M_perspective, (output_width, output_height))

disp_persp_orig = prepare_for_display(img_bgr_with_pts_persp, "Original with Quad Points", target_width=350, target_height=300)
disp_persp_transformed = prepare_for_display(img_perspective_transformed, "Perspective Transformed (Bird-eye)", target_width=350, target_height=300)

combined_perspective = np.hstack((disp_persp_orig, disp_persp_transformed))
cv2.imshow('Perspective Transformation', combined_perspective)
cv2.waitKey(0)
cv2.destroyAllWindows()


# 三、阈值处理

将灰度图像转换为二值图像（只有黑白两色）。（具体作用和目的看我另外一篇博客）

原理：

    cv2.threshold(src, thresh, maxval, type): 全局阈值。
    cv2.adaptiveThreshold(src, maxValue, adaptiveMethod, thresholdType, blockSize, C): 自适应阈值，对光照不均的图像效果更好。

In [ ]:
img_bgr = img_bgr_original.copy()
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
# 应用不同的阈值处理方法
# 简单二值化
ret, thresh_binary = cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY) 

# 自适应阈值处理
# 对光照不均的图像效果更好
thresh_adaptive_mean = cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                           cv2.THRESH_BINARY, 11, 2)
thresh_adaptive_gaussian = cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                               cv2.THRESH_BINARY, 11, 2)

# 准备 2x2 网格显示
w_th, h_th = 250, 200 # 为每个小图设定的宽高
disp_th_gray = prepare_for_display(img_gray, "Original Gray", w_th, h_th)
disp_th_binary = prepare_for_display(thresh_binary, "Simple Binary (T=127)", w_th, h_th)
disp_th_adapt_mean = prepare_for_display(thresh_adaptive_mean, "Adaptive Mean", w_th, h_th)
disp_th_adapt_gauss = prepare_for_display(thresh_adaptive_gaussian, "Adaptive Gaussian", w_th, h_th)

row1 = np.hstack((disp_th_gray, disp_th_binary))
row2 = np.hstack((disp_th_adapt_mean, disp_th_adapt_gauss))
combined_thresholding = np.vstack((row1, row2))

cv2.imshow('Thresholding Techniques', combined_thresholding)
cv2.waitKey(0)
cv2.destroyAllWindows()


# 四、图像平滑/模糊

主要用于减少图像中的噪声。(具体看我另外一篇博客)

原理：

大多基于“卷积”操作，用一个“核”滑过图像。

    均值滤波 (cv2.blur()): 邻域像素平均值。
    高斯滤波 (cv2.GaussianBlur()): 高斯核加权平均，更好地保留边缘。
    中值滤波 (cv2.medianBlur()): 邻域像素中值，对椒盐噪声有效。
    双边滤波 (cv2.bilateralFilter()): 考虑空间距离和像素值差异，去噪同时保边。

In [ ]:
img_bgr = img_bgr_original.copy()

# 给原始 BGR 图像添加高斯噪声
noise = np.zeros(img_bgr.shape, np.uint8)
cv2.randn(noise, (0,0,0), (30,30,30))
img_noisy_bgr = cv2.add(img_bgr, noise)

# 滤波操作
img_mean_blur = cv2.blur(img_noisy_bgr, (5, 5))
img_gaussian_blur = cv2.GaussianBlur(img_noisy_bgr, (5, 5), 0)
img_median_blur = cv2.medianBlur(img_noisy_bgr, 5)
img_bilateral_blur = cv2.bilateralFilter(img_noisy_bgr, 9, 75, 75)

# 准备 2x3 网格显示
w_sm, h_sm = 220, 180 # 调整每个小图的宽高
disp_sm_noisy = prepare_for_display(img_noisy_bgr, "Noisy", w_sm, h_sm)
disp_sm_mean = prepare_for_display(img_mean_blur, "Mean Blur", w_sm, h_sm)
disp_sm_gaussian = prepare_for_display(img_gaussian_blur, "Gaussian Blur", w_sm, h_sm)
disp_sm_median = prepare_for_display(img_median_blur, "Median Blur", w_sm, h_sm)
disp_sm_bilateral = prepare_for_display(img_bilateral_blur, "Bilateral Filter", w_sm, h_sm)
disp_sm_original_clean = prepare_for_display(img_bgr, "Original Clean", w_sm, h_sm)

row_blur1 = np.hstack((disp_sm_noisy, disp_sm_mean, disp_sm_gaussian))
row_blur2 = np.hstack((disp_sm_median, disp_sm_bilateral, disp_sm_original_clean))
combined_smoothing = np.vstack((row_blur1, row_blur2))

cv2.imshow('Image Smoothing (Gaussian Noise)', combined_smoothing)
cv2.waitKey(0)
cv2.destroyAllWindows()

# 示例：添加椒盐噪声并用中值滤波处理
img_salt_pepper_bgr = img_bgr_original.copy()
# 添加盐噪声
num_salt = np.ceil(0.02 * img_salt_pepper_bgr.size * 0.33) # 约2%的像素点，分通道
for _ in range(int(num_salt)):
    i = np.random.randint(0, img_salt_pepper_bgr.shape[0]-1)
    j = np.random.randint(0, img_salt_pepper_bgr.shape[1]-1)
    k = np.random.randint(0, 2) # choose one channel
    img_salt_pepper_bgr[i, j, k] = 255
# 添加胡椒噪声
num_pepper = np.ceil(0.02 * img_salt_pepper_bgr.size * 0.33)
for _ in range(int(num_pepper)):
    i = np.random.randint(0, img_salt_pepper_bgr.shape[0]-1)
    j = np.random.randint(0, img_salt_pepper_bgr.shape[1]-1)
    k = np.random.randint(0,2)
    img_salt_pepper_bgr[i, j, k] = 0

img_median_blur_sp = cv2.medianBlur(img_salt_pepper_bgr, 5)

disp_sp_orig = prepare_for_display(img_bgr_original.copy(), "Original", w_sm, h_sm)
disp_sp_noisy = prepare_for_display(img_salt_pepper_bgr, "Salt & Pepper Noise", w_sm, h_sm)
disp_sp_median_filtered = prepare_for_display(img_median_blur_sp, "Median Filtered", w_sm, h_sm)

combined_sp_noise = np.hstack((disp_sp_orig, disp_sp_noisy, disp_sp_median_filtered))
cv2.imshow('Median Filter for Salt & Pepper Noise', combined_sp_noise)
cv2.waitKey(0)
cv2.destroyAllWindows()


# 霍夫变换与车道线检测
霍夫变换
在图像空间中，一条直线是通过 (x) 和 (y) 的关系来绘制的。但在 1962 年，Paul Hough 提出了一种在参数空间中表示直线的方法，为了纪念他，我们将这个参数空间称为“霍夫空间”。

在霍夫空间中，我们可以将“(x) 与 (y) 的关系”表示为“(m) 与 (b) 的关系”。霍夫变换就是从图像空间到霍夫空间的转换。因此，图像空间中的一条直线在霍夫空间中将被表示为一个点，其位置为 ((m, b))。更多详情可以参考 霍夫变换教程。

In [ ]:
# Read in and grayscale the image
image = img_bgr_original.copy()
gray = cv2.cvtColor(image,cv2.COLOR_RGB2GRAY)

# Define a kernel size and apply Gaussian smoothing
kernel_size = 5
blur_gray = cv2.GaussianBlur(gray,(kernel_size, kernel_size),0)

# Define our parameters for Canny and apply
low_threshold = 180
high_threshold = 240
edges = cv2.Canny(blur_gray, low_threshold, high_threshold)

# Next we'll create a masked edges image using cv2.fillPoly()
mask = np.zeros_like(edges)   
ignore_mask_color = 255   

# This time we are defining a four sided polygon to mask
imshape = image.shape
vertices = np.array([[(0,imshape[0]),(450, 290), (490, 290), (imshape[1],imshape[0])]], dtype=np.int32)
cv2.fillPoly(mask, vertices, ignore_mask_color)
masked_edges = cv2.bitwise_and(edges, mask)

# Define the Hough transform parameters
# Make a blank the same size as our image to draw on
rho = 1 # distance resolution in pixels of the Hough grid
theta = np.pi/180 # angular resolution in radians of the Hough grid
threshold = 2     # minimum number of votes (intersections in Hough grid cell)
min_line_length = 4 #minimum number of pixels making up a line
max_line_gap = 5    # maximum gap in pixels between connectable line segments
line_image = np.copy(image)*0 # creating a blank to draw lines on

# Run Hough on edge detected image
# Output "lines" is an array containing endpoints of detected line segments
lines = cv2.HoughLinesP(masked_edges, rho, theta, threshold, np.array([]),
                            min_line_length, max_line_gap)

# Iterate over the output "lines" and draw lines on a blank image
for line in lines:
    for x1,y1,x2,y2 in line:
        cv2.line(line_image,(x1,y1),(x2,y2),(255,0,0),10)

# Create a "color" binary image to combine with line image
color_edges = np.dstack((edges, edges, edges)) 

# Draw the lines on the edge image
lines_edges = cv2.addWeighted(color_edges, 0.8, line_image, 1, 0)
lines_edges = cv2.polylines(lines_edges,vertices, True, (0,0,255), 10)

# 准备 2x3 网格显示
w_sm, h_sm = 220, 180 # 调整每个小图的宽高
disp_original = prepare_for_display(image, "Original", w_sm, h_sm)
disp_gray = prepare_for_display(gray, "Gray", w_sm, h_sm)
disp_gaussian = prepare_for_display(blur_gray, "Gaussian Blur", w_sm, h_sm)
disp_edges = prepare_for_display(edges, "Edge detection by Cannly", w_sm, h_sm)
disp_masked_edges = prepare_for_display(masked_edges, "Masked Edges", w_sm, h_sm)
disp_lines_edges = prepare_for_display(lines_edges, "Line Detection by Hough", w_sm, h_sm)

row_step1 = np.hstack((disp_original, disp_gray, disp_gaussian))
row_step2 = np.hstack((disp_edges, disp_masked_edges, disp_lines_edges))
combined_disp = np.vstack((row_step1, row_step2))

cv2.imshow('Hough on Lane Line Detection Steps', combined_disp)
cv2.waitKey(0)
cv2.destroyAllWindows()



# 5. Let's Make a Lane Detection Pipeline
### Gray Scale
### Gaussian Smoothing
### Canny Edge Detection
### Region Masking
### Hough Transform
### Draw Lines [Mark Lane Lines with different Color]

In [ ]:
import math

def grayscale(img):
    """Applies the Grayscale transform
    This will return an image with only one color channel
    but NOTE: to see the returned image as grayscale
    (assuming your grayscaled image is called 'gray')
    you should call plt.imshow(gray, cmap='gray')"""
    return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # Or use BGR2GRAY if you read an image with cv2.imread()
    # return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
def canny(img, low_threshold, high_threshold):
    """Applies the Canny transform"""
    return cv2.Canny(img, low_threshold, high_threshold)

def gaussian_blur(img, kernel_size):
    """Applies a Gaussian Noise kernel"""
    return cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)

def region_of_interest(img, vertices):
    """
    Applies an image mask.
    
    Only keeps the region of the image defined by the polygon
    formed from `vertices`. The rest of the image is set to black.
    `vertices` should be a numpy array of integer points.
    """
    #defining a blank mask to start with
    mask = np.zeros_like(img)   
    
    #defining a 3 channel or 1 channel color to fill the mask with depending on the input image
    if len(img.shape) > 2:
        channel_count = img.shape[2]  # i.e. 3 or 4 depending on your image
        ignore_mask_color = (255,) * channel_count
    else:
        ignore_mask_color = 255
        
    #filling pixels inside the polygon defined by "vertices" with the fill color    
    cv2.fillPoly(mask, vertices, ignore_mask_color)
    
    #returning the image only where mask pixels are nonzero
    masked_image = cv2.bitwise_and(img, mask)
    return masked_image


def draw_lines(img, lines, color=[255, 0, 0], thickness=10):
    """
    NOTE: this is the function you might want to use as a starting point once you want to 
    average/extrapolate the line segments you detect to map out the full
    extent of the lane (going from the result shown in raw-lines-example.mp4
    to that shown in P1_example.mp4).  
    
    Think about things like separating line segments by their 
    slope ((y2-y1)/(x2-x1)) to decide which segments are part of the left
    line vs. the right line.  Then, you can average the position of each of 
    the lines and extrapolate to the top and bottom of the lane.
    
    This function draws `lines` with `color` and `thickness`.    
    Lines are drawn on the image inplace (mutates the image).
    If you want to make the lines semi-transparent, think about combining
    this function with the weighted_img() function below
    """
    for line in lines:
        for x1,y1,x2,y2 in line:
            cv2.line(img, (x1, y1), (x2, y2), color, thickness)


def slope_lines(image, lines):
    img = image.copy()
    poly_vertices = []
    order = [0,1,3,2]

    left_lines = [] # Like /
    right_lines = [] # Like \
    
    if lines is not None:  # 添加对lines是否为None的检查
        for line in lines:
            for x1,y1,x2,y2 in line:
                if x1 == x2:
                    continue # 跳过垂直线
                else:
                    m = (y2 - y1) / (x2 - x1)
                    c = y1 - m * x1

                    if m < 0:
                        left_lines.append((m,c))
                    elif m >= 0:
                        right_lines.append((m,c))

    # 处理空列表情况
    if len(left_lines) > 0:
        left_line = np.mean(left_lines, axis=0)
    else:
        left_line = None
        
    if len(right_lines) > 0:
        right_line = np.mean(right_lines, axis=0)
    else:
        right_line = None

    # 只处理有效的线
    valid_lines = []
    if left_line is not None:
        valid_lines.append(left_line)
    if right_line is not None:
        valid_lines.append(right_line)

    for slope, intercept in valid_lines:
        rows, cols = image.shape[:2]
        y1 = int(rows) # 图像底部
        y2 = int(rows*0.6) # 图像60%高度
        
        try:
            x1 = int((y1-intercept)/slope)
            x2 = int((y2-intercept)/slope)
            poly_vertices.append((x1, y1))
            poly_vertices.append((x2, y2))
            draw_lines(img, np.array([[[x1,y1,x2,y2]]]))
        except:
            continue
    
    if len(poly_vertices) >= 4:  # 确保有足够的点绘制多边形
        poly_vertices = [poly_vertices[i] for i in order]
        cv2.fillPoly(img, pts = np.array([poly_vertices],'int32'), color = (0,255,0))
    
    return cv2.addWeighted(image,0.7,img,0.4,0.)

def hough_lines(img, rho, theta, threshold, min_line_len, max_line_gap):
    lines = cv2.HoughLinesP(img, rho, theta, threshold, np.array([]), 
                          minLineLength=min_line_len, maxLineGap=max_line_gap)
    
    line_img = np.zeros((img.shape[0], img.shape[1], 3), dtype=np.uint8)
    if lines is not None and len(lines) > 0:  # 确保有检测到线
        line_img = slope_lines(line_img, lines)
    return line_img

# Python 3 has support for cool math symbols.

def weighted_img(img, initial_img, α=0.1, β=1., γ=0.):
    """
    `img` is the output of the hough_lines(), An image with lines drawn on it.
    Should be a blank image (all black) with lines drawn on it.
    
    `initial_img` should be the image before any processing.
    
    The result image is computed as follows:
    
    initial_img * α + img * β + γ
    NOTE: initial_img and img must be the same shape!
    """
    lines_edges = cv2.addWeighted(initial_img, α, img, β, γ)
    #lines_edges = cv2.polylines(lines_edges,get_vertices(img), True, (0,0,255), 10)
    return lines_edges
def get_vertices(image):
    rows, cols = image.shape[:2]
    bottom_left  = [cols*0.15, rows]
    top_left     = [cols*0.45, rows*0.6]
    bottom_right = [cols*0.95, rows]
    top_right    = [cols*0.55, rows*0.6] 
    
    ver = np.array([[bottom_left, top_left, top_right, bottom_right]], dtype=np.int32)
    return ver

In [ ]:
# Lane finding Pipeline
def lane_finding_pipeline(image):
    
    #Grayscale
    gray_img = grayscale(image)
    #Gaussian Smoothing
    smoothed_img = gaussian_blur(img = gray_img, kernel_size = 5)
    #Canny Edge Detection
    canny_img = canny(img = smoothed_img, low_threshold = 180, high_threshold = 240)
    #Masked Image Within a Polygon
    masked_img = region_of_interest(img = canny_img, vertices = get_vertices(image))
    #Hough Transform Lines
    houghed_lines = hough_lines(img = masked_img, rho = 1, theta = np.pi/180, threshold = 20, min_line_len = 20, max_line_gap = 180)
    #Draw lines on edges
    output = weighted_img(img = houghed_lines, initial_img = image, α=0.8, β=1., γ=0.)
    
    return output

# Test our Algorithm Pipeline with different Images

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg  # 导入 mpimg 用于读取图像

# 定义图像文件夹路径
image_folder = r'D:\AIR\Teaching\Lane Line Detection\test_images'

# 遍历文件夹中的所有图像文件
for image_filename in os.listdir(image_folder):
    # 创建完整图像路径
    image_path = os.path.join(image_folder, image_filename)
    
    # 读取图像
    image = mpimg.imread(image_path)
    
    # 创建画布并设置大小
    fig = plt.figure(figsize=(20, 10))
    
    # 添加第一个子图显示原图
    ax1 = fig.add_subplot(1, 2, 1)
    plt.imshow(image)
    ax1.set_title("Input Image")
    ax1.set_xticks([])  # 隐藏x轴刻度
    ax1.set_yticks([])  # 隐藏y轴刻度

    # 应用车道线检测算法（确保 lane_finding_pipeline 函数已定义并导入）
    output_image = lane_finding_pipeline(image)
    
    # 添加第二个子图显示处理后的图像
    ax2 = fig.add_subplot(1, 2, 2)
    plt.imshow(output_image)
    ax2.set_title("Output Image [Lane Line Detected]")
    ax2.set_xticks([])  # 隐藏x轴刻度
    ax2.set_yticks([])  # 隐藏y轴刻度

    # 显示图像
    plt.show()

# 6. Let's try with Video Stream [Yes! Real-time Lane Line Detection]

In [ ]:
import cv2
import numpy as np
import os
from IPython.display import display, HTML


# 定义视频输入和输出路径
input_video_path = r'D:\AIR\Teaching\Lane Line Detection\test_videos\challenge.mp4'
output_video_path = r'D:\AIR\Teaching\Lane Line Detection\output\Process_output.mp4'

# 创建输出目录（如果不存在）
os.makedirs(os.path.dirname(output_video_path), exist_ok=True)

# 获取视频属性
cap = cv2.VideoCapture(input_video_path)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# 定义视频编码器和创建VideoWriter对象
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # 对于 MP4 文件
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# 遍历视频的每一帧
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 应用车道线检测函数
    processed_frame = lane_finding_pipeline(frame)

    # 将处理后的帧写入输出视频
    out.write(processed_frame)

# 释放资源
cap.release()
out.release()

print("Video processing complete.")

# 7. 展示检测结果视频流

In [22]:
# 打开视频文件
cap = cv2.VideoCapture(output_video_path)

# 检查视频是否成功打开
if not cap.isOpened():
    print("Error: Could not open video.")
    exit()

# 获取视频帧率
fps = cap.get(cv2.CAP_PROP_FPS)
# 计算每帧展示的时间间隔（毫秒）
frame_delay = int(1000 / fps)

# 读取视频帧并显示
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # 显示每一帧
    cv2.imshow('Video Playback', frame)
    
    # 按照视频帧率设置延迟，并允许按下任意键退出
    if cv2.waitKey(frame_delay) & 0xFF != 255:
        break

# 释放视频捕获器资源
cap.release()
cv2.destroyAllWindows()